In [1]:
from __future__ import print_function, division
import os, random, time, copy
import numpy as np
import pandas as pd
import os.path as path
import scipy.io as sio
from scipy import misc
from scipy import ndimage, signal
import scipy
import pickle
import sys
import math
import matplotlib.pyplot as plt
from io import BytesIO


import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler 
import torch.nn.functional as F
from torch.autograd import Variable
import torchvision
from torchvision import datasets, models, transforms
import torchvision.utils as vutils


import warnings 
warnings.filterwarnings("ignore")
print(sys.version)
print(torch.__version__)


manualSeed = 999
print("Random Seed: ", manualSeed)
random.seed(manualSeed)
torch.manual_seed(manualSeed)

3.9.18 (main, Sep 11 2023, 14:09:26) [MSC v.1916 64 bit (AMD64)]
1.12.1
Random Seed:  999


In [2]:
torch.manual_seed(0)

device ='cpu'
if torch.cuda.is_available(): 
    device='cuda:0'


batch_size = 16    
bestEpoch = 4  
lr = 0.0001 
num_epochs = 1
torch.cuda.device_count()
torch.cuda.empty_cache()

save_dir="./savedata/RPGAN"

print(save_dir)    
if not os.path.exists(save_dir): os.makedirs(save_dir)

log_filename = os.path.join(save_dir, 'train.log')

./savedata/RPGAN


In [3]:

print('==> Preparing data.............................')


import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import os
import torch
import numpy as np
from PIL import Image
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torchvision.datasets import MNIST, CIFAR10, CIFAR100, SVHN

import os
import torch
from torch import nn,optim
import torch.nn.functional as F
from torchvision import datasets, transforms

from time import perf_counter

import  numpy as np
import torch.utils.data as Data

from torch.utils.data import Dataset, DataLoader

class Safeman(Dataset):
    
    def __init__(self, data,targets):
        super(Safeman, self).__init__()
        self.data = data
        self.targets = targets
        
     
    def __len__(self):
        return len(self.targets)

    def __getitem__(self, idx):
        img, target = self.data[idx], self.targets[idx]
        return img, target

class Safeman_Filter(Safeman):

    def __Filter__(self, known):
        targets = self.targets.data.numpy()
        mask, new_targets = [], []
        for i in range(len(targets)):
            if targets[i] in known:
                mask.append(i)
                new_targets.append(known.index(targets[i]))
        self.targets = np.array(new_targets)
        mask = torch.tensor(mask).long()
        self.data = torch.index_select(self.data, 0, mask)
        
class Safeman_FilterB(Safeman):

    def __Filter__(self, known):
        targets = self.targets.data.numpy()
        new_targets = []
        for i in range(len(targets)):
            if targets[i] in known:
                new_targets.append(0)
            else:
                new_targets.append(1)
        self.targets = np.array(new_targets)
        self.data = self.data

class Safeman_FilterC(Safeman):
    
    def __Filter__(self, trainknown):
        train_class_num=len(trainknown)
        for i in range(0,len(self.targets)) :
            if self.targets[i]>train_class_num:
                self.targets[i] = train_class_num
        self.data = self.data

        
class Safeman_FilterD(Safeman):

    def __Filter__(self, known):
        targets = self.targets.data.numpy()
        mask, new_targets = [], []
        train_class_num=len(known)
        for i in range(len(targets)):
            if targets[i] not in known:
                mask.append(i)
                new_targets.append(train_class_num)
        self.targets = np.array(new_targets)
        mask = torch.tensor(mask).long()
        self.data = torch.index_select(self.data, 0, mask)


        
def setup_seed(seed):

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    torch.backends.cudnn.deterministic = True

setup_seed(8)



known=[0, 1, 2]
unknown=[3,4]

num_class=len(known)



X_train0 = np.load('C:/Users/lenovo/Desktop/我的论文/论文4/数据集/X_train.npy')
y_train1 = np.load('C:/Users/lenovo/Desktop/我的论文/论文4/数据集/y_train.npy')
X_final_test0 = np.load('C:/Users/lenovo/Desktop/我的论文/论文4/数据集/X_test.npy' )
y__final_test1 = np.load('C:/Users/lenovo/Desktop/我的论文/论文4/数据集/y_test.npy')


X_train1=[]
X_final_test1=[]

for i in range(len(y_train1)):
    a = np.resize(X_train0[i], (3, 8, 8))
    X_train1 += [a]
    
for j in range(len(y__final_test1)):
    b = np.resize(X_final_test0[j], (3, 8, 8))
    X_final_test1 += [b]

i=0
j=0



x_train, x_test, y_train,y_test = torch.Tensor(X_train1), torch.Tensor(X_final_test1), torch.from_numpy(y_train1), torch.from_numpy(y__final_test1)

print(x_train.shape, x_test.shape, y_train.shape,y_test.shape)

train_dataset = Data.TensorDataset(x_train, y_train)
train_dataset.data = train_dataset.tensors[0]
train_dataset.targets = train_dataset.tensors[1]



test_dataset = Data.TensorDataset(x_test, y_test)
test_dataset.data = test_dataset.tensors[0]
test_dataset.targets = test_dataset.tensors[1]



labels =['Normal', 'RPM', 'gear', 'DoS', 'Fuzzy']



train_dataset.classes = labels
test_dataset.classes = labels

train_dataset.classes_to_idx = {i: label for i, label in enumerate(labels)}
test_dataset.classes_to_idx = {i: label for i, label in enumerate(labels)}



b_s=16

trainset = Safeman_Filter(data=train_dataset.data,targets=train_dataset.targets)
print('All down Train Data:', len(trainset))
trainset.__Filter__(known=known)


train_loader = torch.utils.data.DataLoader(
    trainset, batch_size=b_s, shuffle=True,
    num_workers=0,drop_last=True)


print('Real train Data:', len(trainset))



testsetA = Safeman_Filter(data=test_dataset.data,targets=test_dataset.targets)
print('All testsetA Data:', len(testsetA))
testsetA.__Filter__(known=known)


test_loader_A = torch.utils.data.DataLoader(
    testsetA, batch_size=b_s, shuffle=False,
    num_workers=0,drop_last=True)


print('Real testsetA Data:', len(testsetA))


testsetD = Safeman_FilterD(data=test_dataset.data,targets=test_dataset.targets)
print('All testsetA Data:', len(testsetA))
testsetD.__Filter__(known=known)


test_loader_D = torch.utils.data.DataLoader(
    testsetD, batch_size=b_s, shuffle=False,
    num_workers=0,drop_last=True)


print('Real testsetD Data:', len(testsetD))


testsetC = Safeman_FilterC(data=test_dataset.data,targets=test_dataset.targets)
print('All testsetC Data:', len(testsetC))
testsetC.__Filter__(trainknown=known)


test_loader_C = torch.utils.data.DataLoader(
    testsetC, batch_size=b_s, shuffle=False,
    num_workers=0,drop_last=True)


print('Real testsetC Data:', len(testsetC))


print("done!")

==> Preparing data.............................
torch.Size([632087, 3, 8, 8]) torch.Size([158022, 3, 8, 8]) torch.Size([632087]) torch.Size([158022])
All down Train Data: 632087
Real train Data: 588729
All testsetA Data: 158022
Real testsetA Data: 147255
All testsetA Data: 147255
Real testsetD Data: 10767
All testsetC Data: 158022
Real testsetC Data: 158022
done!


In [4]:
      
class Generatorzy(nn.Module):
    def __init__(self, z_dim):
        super(Generatorzy, self).__init__()
        
        self.fc = nn.Linear(z_dim, 128*2*2)  # 修改输出维度以适应 8x8 的输入
        self.g_deconv_1 = nn.Sequential(
                          nn.ConvTranspose2d(128, 64, kernel_size=4,
                                    stride=2, padding=1),
                          nn.BatchNorm2d(64),
                          nn.LeakyReLU()
                          )
        self.g_deconv_2 = nn.Sequential(
                          nn.ConvTranspose2d(64, 32, kernel_size=4,
                                    stride=2, padding=1),
                          nn.BatchNorm2d(32),
                          nn.LeakyReLU()
                          )
        self.g_deconv_3 = nn.Sequential(
                          nn.ConvTranspose2d(32, 3, kernel_size=3,
                                    stride=1, padding=1),
                          nn.Tanh()
                          )

    def forward(self, x):
        x = self.fc(x).view(-1, 128, 2, 2)  # 修改形状以适应新的全连接层输出
        x = self.g_deconv_1(x)
        x = self.g_deconv_2(x)
        x = self.g_deconv_3(x)
        
        return x


    
    


    
class Discriminatorzy(nn.Module):
    def __init__(self):
        super(Discriminatorzy, self).__init__()
        
        self.d_conv_1 = nn.Sequential(
                          nn.Conv2d(3, 32, kernel_size=4,
                                    stride=2, padding=1),
                          nn.LeakyReLU()
                          )
        self.d_conv_2 = nn.Sequential(
                          nn.Conv2d(32, 64, kernel_size=4,
                                    stride=2, padding=1),
                          nn.LeakyReLU()
                          )
        self.d_conv_3 = nn.Sequential(
                          nn.Conv2d(64, 128, kernel_size=3,
                                    stride=2, padding=1),
                          nn.LeakyReLU()
                          )
        self.fc = nn.Linear(128*1*1, 1)  # 修改全连接层输入维度
        self.fczy = nn.Linear(128*1*1, 64)  # 修改全连接层输入维度

    def forward(self, x):
        x = self.d_conv_1(x)
        x = self.d_conv_2(x)
        x = self.d_conv_3(x)
        x = x.view(-1, 128*1*1)
        x_zy = self.fczy(x)
        x = torch.sigmoid(self.fc(x))
        return x_zy, x





In [5]:
print(device)

def weights_init(m):
    classname = m.__class__.__name__
    if classname.find('Conv') != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find('BatchNorm') != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)     
        



z_dim=100
batch_size=b_s
discriminator = Discriminatorzy()
netDzy = discriminator.to(device)

discriminatoraux = Discriminatorzy()
netDzyaux = discriminatoraux.to(device)

generator = Generatorzy(z_dim)
netGzy = generator.to(device)


netDzy.apply(weights_init)

netDzyaux.apply(weights_init)

netGzy.apply(weights_init)


criterion = nn.BCELoss()

real_label = 1
fake_label = 0
beta1 = 0.5

optimizerD = optim.Adam(netDzy.parameters(), lr=lr/1.5, betas=(beta1, 0.999))

optimizerDaux = optim.Adam(netDzyaux.parameters(), lr=lr/1.5, betas=(beta1, 0.999))

optimizerG = optim.Adam(netGzy.parameters(), lr=lr, betas=(beta1, 0.999))

cuda:0


In [6]:

print("Starting Training Loop...")

dataloader_train_closeset=train_loader

for epoch in range(num_epochs):
    i=0
    for sample in dataloader_train_closeset:
        
        data, datalabel = sample
        ############################
        # (1) Update D1 D2 network
        ###########################
        
        netDzy.zero_grad()
        netDzyaux.zero_grad()
        
        real_cpu = data.to(device)
        bb_size = real_cpu.size(0)
        
        label = torch.full((bb_size,), real_label, device=device)
        labelaux = torch.full((bb_size,), fake_label, device=device)
        
        _,output = netDzy(real_cpu)
        output=output.view(-1)
        
        _,outputaux = netDzyaux(real_cpu)
        outputaux=outputaux.view(-1)
        
        output=output.to(torch.float32)
        outputaux=outputaux.to(torch.float32) 
    
        label=label.to(torch.float32)
        labelaux=labelaux.to(torch.float32)
        
        errD_real = criterion(output, label)        
        errDaux_real = criterion(outputaux, labelaux)

        errD_real.backward()        
        errDaux_real.backward()
        

        noise = torch.randn(batch_size, z_dim, device=device)

        fake = netGzy(noise)
        label.fill_(fake_label)
        
        labelaux.fill_(real_label)

        _,output = netDzy(fake.detach())
        
        output=output.view(-1)

        output=output.to(torch.float32)
        
        _,outputaux = netDzyaux(fake.detach())
        outputaux=outputaux.view(-1)
        
        outputaux=outputaux.to(torch.float32)
        
        label=label.to(torch.float32)
        labelaux=labelaux.to(torch.float32)
        
        errD_fake = criterion(output, label)
        
        errDaux_fake = criterion(outputaux, labelaux)
        
        errD_fake.backward()
        
        errDaux_fake.backward()
        
        
        optimizerD.step()
        optimizerDaux.step()
        

        ############################
        # (2) Update G network
        ###########################
        netGzy.zero_grad()
        label.fill_(real_label)  
        
        labelaux.fill_(fake_label) 
        

        _,output = netDzy(fake)
        output=output.view(-1)

        output=output.to(torch.float32)
        label=label.to(torch.float32)
        errG = criterion(output, label)
        

        _,outputaux = netDzyaux(fake)
        outputaux=outputaux.view(-1)

        outputaux=outputaux.to(torch.float32)
        labelaux=labelaux.to(torch.float32)
        errGaux = criterion(outputaux, labelaux)
        
        


        errGqiuhe=errG-errGaux*0.1  
        errGqiuhe.backward()


        optimizerG.step()
          
            
        if i % 500 == 0:
            print('[%d/%d][%d/%d]\terrD_real: %.4f\terrDaux_real: %.4f\terrD_fake: %.4f\terrDaux_fake: %.4f \terrGqiuhe: %.4f'
                  % (epoch, num_epochs, i, len(dataloader_train_closeset),
                     errD_real, errDaux_real,errD_fake,errDaux_fake, errGqiuhe))
        i+=1
        
        
    cur_model_wts = copy.deepcopy(netGzy.state_dict())
    path_to_save_paramOnly = os.path.join(save_dir, 'RPGAN-epoch-{}.GNet'.format(epoch+1))
    torch.save(cur_model_wts, path_to_save_paramOnly)
    
    cur_model_wts = copy.deepcopy(netDzy.state_dict())
    path_to_save_paramOnly = os.path.join(save_dir, 'RPGAN-epoch-{}.DNet'.format(epoch+1))
    torch.save(cur_model_wts, path_to_save_paramOnly)
    
   
    cur_model_wts = copy.deepcopy(netDzyaux.state_dict())
    path_to_save_paramOnly = os.path.join(save_dir, 'RPGAN-epoch-{}.DauxNet'.format(epoch+1))
    torch.save(cur_model_wts, path_to_save_paramOnly)

Starting Training Loop...
[0/1][0/36795]	errD_real: 0.8754	errDaux_real: 0.6972	errD_fake: 0.6823	errDaux_fake: 0.7118 	errGqiuhe: 0.6362
[0/1][50/36795]	errD_real: 0.1011	errDaux_real: 0.1080	errD_fake: 0.6908	errDaux_fake: 0.7099 	errGqiuhe: 0.6281
[0/1][100/36795]	errD_real: 0.0609	errDaux_real: 0.0585	errD_fake: 0.6573	errDaux_fake: 0.6568 	errGqiuhe: 0.6582
[0/1][150/36795]	errD_real: 0.0556	errDaux_real: 0.0537	errD_fake: 0.6201	errDaux_fake: 0.5294 	errGqiuhe: 0.6842
[0/1][200/36795]	errD_real: 0.0996	errDaux_real: 0.1099	errD_fake: 0.5723	errDaux_fake: 0.2605 	errGqiuhe: 0.6840
[0/1][250/36795]	errD_real: 0.0112	errDaux_real: 0.0122	errD_fake: 0.4938	errDaux_fake: 0.3438 	errGqiuhe: 0.8253
[0/1][300/36795]	errD_real: 0.0681	errDaux_real: 0.0757	errD_fake: 0.3351	errDaux_fake: 0.2964 	errGqiuhe: 1.1336
[0/1][350/36795]	errD_real: 0.0425	errDaux_real: 0.0415	errD_fake: 0.1449	errDaux_fake: 0.0337 	errGqiuhe: 1.6808
[0/1][400/36795]	errD_real: 0.0037	errDaux_real: 0.0034	errD_fake

In [11]:
print(device)

bestEpoch =1 
z_dim=100

discriminator = Discriminatorzy()
netDzy = discriminator.to(device)
generator = Generatorzy(z_dim)
netGzy = generator.to(device)


path_to_G = os.path.join(save_dir, 'RPGAN-epoch-{}.GNet'.format(bestEpoch))
path_to_D = os.path.join(save_dir, 'RPGAN-epoch-{}.DNet'.format(bestEpoch))
netGzy.load_state_dict(torch.load(path_to_G))
netDzy.load_state_dict(torch.load(path_to_D))


cuda:0


<All keys matched successfully>

In [12]:

featesttrain=[]

for sample in train_loader:    
    datazy, label = sample
    datazy = datazy.to(device)
    label = label.type(torch.long).view(-1).to(device)    
    features1,_ = netDzy(datazy) 
    featesttrain+=features1

torch.save(featesttrain,'./savedata/featesttrain0102.pkl')

In [13]:
featestA=[]

dataloader_test_closeset= test_loader_A

for sample in dataloader_test_closeset:    
    datazy, label = sample
    datazy = datazy.to(device)
    label = label.type(torch.long).view(-1).to(device)    
    features1,_ = netDzy(datazy)
    featestA+=features1


torch.save(featestA,'./savedata/featestA0102.pkl')

In [6]:
featestD=[]

dataloader_openset = test_loader_D

for sample in dataloader_openset:
    datazy, label = sample
    datazy = datazy.to(device)    
    features1,_ = netDzy(datazy) 
    featestD+=features1


torch.save(featestD,'./savedata/featestD0102.pkl')


print("save done!")

save done!
